<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/notebooks/Streamlit_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install streamlit pyngrok -q

print("Streamlit and ngrok installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.3 MB/s eta 0:00:00
Streamlit and ngrok installed!


In [3]:
from pyngrok import ngrok

ngrok.set_auth_token("3BnqLvHeZ3gddavL9xtNePRzixY_5BufFg3xV4vxMjXW7Gru5")

print("ngrok configured!")

ngrok configured!


In [4]:
import pickle

# ✅ Correct path from Drive
path = '/content/drive/MyDrive/fraud_detection_project/federated_model.pkl'

raw_models = pickle.load(open(path, 'rb'))

# If it's a list → take first model
if isinstance(raw_models, list):
    final_model = raw_models[0]
else:
    final_model = raw_models

# Save back
pickle.dump(final_model, open(path, 'wb'))

# Verify
check = pickle.load(open(path, 'rb'))
print(type(check))

print("✅ Model fixed!")

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
✅ Model fixed!


In [5]:
%%writefile /content/app.py

import streamlit as st
import numpy as np
import pickle
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ─── PAGE CONFIG ───
st.set_page_config(
    page_title="FraudShield — AI Fraud Detection",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ─── CUSTOM CSS ───
st.markdown("""
""", unsafe_allow_html=True)

# ─── LOAD ALL RESOURCES ───
@st.cache_resource
def load_all():
    base = '/content/drive/MyDrive/fraud_detection_project/'

    raw = pickle.load(open(base + 'federated_model.pkl', 'rb'))
    if isinstance(raw, list):
        fed_models = raw
    else:
        fed_models = [raw]

    scaler   = pickle.load(open(base + 'scaler.pkl', 'rb'))
    features = pickle.load(open(base + 'feature_columns.pkl', 'rb'))
    medians  = pickle.load(open(base + 'feature_medians.pkl', 'rb'))

    return fed_models, scaler, features, medians

# ─── LOAD MODELS ───
try:
    fed_models, scaler, feature_columns, feature_medians = load_all()
    model_loaded = True
    num_models   = len(fed_models)
    num_features = len(feature_columns)
    THRESHOLD    = 0.5
except Exception as e:
    model_loaded  = False
    num_models    = 3
    num_features  = 224
    THRESHOLD     = 0.5
    st.error(f"Model loading error: {e}")

# ─── FEDERATED PREDICTION ───
def federated_predict(fed_models, input_vector, threshold=0.5):
    all_probs = []
    for m in fed_models:
        probs = m.predict_proba(input_vector)[:, 1]
        all_probs.append(probs)
    avg_prob   = np.mean(all_probs, axis=0)
    prediction = (avg_prob >= threshold).astype(int)
    return prediction[0], float(avg_prob[0])

# ─── BUILD INPUT VECTOR ───
def build_input_vector(user_inputs, feature_columns, feature_medians):
    input_vector = {col: feature_medians.get(col, 0) for col in feature_columns}

    card4_map   = {'Visa': 0, 'Mastercard': 1, 'American Express': 2, 'Discover': 3}
    card6_map   = {'Debit': 0, 'Credit': 1}
    product_map = {'W': 0, 'H': 1, 'C': 2, 'S': 3, 'R': 4}
    email_map   = {'gmail.com': 0, 'yahoo.com': 1, 'hotmail.com': 2,
                   'outlook.com': 3, 'other': 4}
    match_map   = {'Yes': 1, 'No': 0}

    if 'TransactionAmt' in input_vector:
        input_vector['TransactionAmt'] = user_inputs['amount']
    if 'card4' in input_vector:
        input_vector['card4'] = card4_map.get(user_inputs['card_network'], 0)
    if 'card6' in input_vector:
        input_vector['card6'] = card6_map.get(user_inputs['card_type'], 0)
    if 'ProductCD' in input_vector:
        input_vector['ProductCD'] = product_map.get(user_inputs['product'], 0)
    if 'P_emaildomain' in input_vector:
        input_vector['P_emaildomain'] = email_map.get(user_inputs['email_domain'], 4)
    if 'TransactionDT' in input_vector:
        input_vector['TransactionDT'] = user_inputs['hour'] * 3600
    if 'M1' in input_vector:
        input_vector['M1'] = match_map.get(user_inputs['name_match'], 1)
    if 'M2' in input_vector:
        input_vector['M2'] = match_map.get(user_inputs['addr_match'], 1)
    if 'M3' in input_vector:
        input_vector['M3'] = match_map.get(user_inputs['zip_match'], 1)
    if 'dist1' in input_vector:
        input_vector['dist1'] = user_inputs['distance']

    arr = np.array([input_vector[col] for col in feature_columns]).reshape(1, -1)
    return arr

# ════════════════════════════════════════
# HERO HEADER
# ════════════════════════════════════════
st.markdown('🛡️ FraudShield', unsafe_allow_html=True)
st.markdown(
    'AI-powered fraud detection using Federated Learning + Homomorphic Encryption',
    unsafe_allow_html=True
)
st.markdown("")

# ─── TOP STATS ───
c1, c2, c3, c4, c5 = st.columns(5)

for col, val, label in [
    (c1, "3", "Banks Connected"),
    (c2, "5", "Training Rounds"),
    (c3, "0.88", "ROC-AUC Score"),
    (c4, "55%", "Fraud Recall"),
    (c5, "128-bit", "Encryption"),
]:
    with col:
        st.markdown(f"""
        {val}
        {label}
        """, unsafe_allow_html=True)

st.markdown("---")

# ════════════════════════════════════════
# TABS
# ════════════════════════════════════════
tab1, tab2, tab3 = st.tabs([
    "🔍  Fraud Detector",
    "📊  Model Dashboard",
    "🔐  Privacy & Security"
])

# ════════════════════════════════════════
# TAB 1 — FRAUD DETECTOR
# ════════════════════════════════════════
with tab1:
    st.markdown('Transaction Fraud Detector', unsafe_allow_html=True)

    st.caption(
        f"Fill in 10 transaction details. Our federated model uses these + "
        f"{num_features - 10} auto-filled features to predict fraud."
    )

    st.markdown("")

    with st.form("fraud_form"):
        st.markdown("#### Enter Transaction Details")

        r1c1, r1c2, r1c3 = st.columns(3)
        r2c1, r2c2, r2c3 = st.columns(3)
        r3c1, r3c2, r3c3 = st.columns(3)
        r4c1, _ = st.columns([1, 2])

        with r1c1:
            amount = st.number_input("💰 Transaction Amount (₹)", 0.0, 1000000.0, 5000.0, 500.0)

        with r1c2:
            card_network = st.selectbox("💳 Card Network",
                                       ["Visa", "Mastercard", "American Express", "Discover"])

        with r1c3:
            card_type = st.selectbox("🏦 Card Type", ["Debit", "Credit"])

        with r2c1:
            product = st.selectbox("🛍️ Product Type",
                                   ["W", "H", "C", "S", "R"],
                                   help="W=Web, H=Home, C=Cash, S=Services, R=Retail")

        with r2c2:
            email_domain = st.selectbox("📧 Buyer Email Domain",
                                       ["gmail.com", "yahoo.com", "hotmail.com",
                                        "outlook.com", "other"])

        with r2c3:
            hour = st.slider("🕐 Transaction Hour", 0, 23, 14)

        with r3c1:
            name_match = st.radio("👤 Name Matches?", ["Yes", "No"], horizontal=True)

        with r3c2:
            addr_match = st.radio("🏠 Address Matches?", ["Yes", "No"], horizontal=True)

        with r3c3:
            zip_match = st.radio("📮 Zip Code Matches?", ["Yes", "No"], horizontal=True)

        with r4c1:
            distance = st.number_input("📍 Distance Between Addresses (km)",
                                       0.0, 5000.0, 0.0, 10.0)

        st.markdown("")
        submitted = st.form_submit_button("🔍 Analyze Transaction",
                                          type="primary",
                                          use_container_width=True)

Writing /content/app.py


In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

ngrok.kill()
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(6)

public_url = ngrok.connect(8501)
print("=" * 50)
print(f"🛡️  FraudShield LIVE at: {public_url}")
print("=" * 50)